In [71]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score



In [72]:
data = pd.read_csv("../data/kaggle_b2_fraud_train_v3.csv")

In [73]:
df = data.copy()
df

,customer_id,account_id,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,...,internal_signal_5,internal_signal_6,internal_signal_7,internal_signal_8,terms_accepted_flag,partner_risk_indicator,manual_review_result,post_event_status_code,chargeback_resolution_time_days,legacy_partner_score
0,CUST_6O9Q8D4I36,ACC_TXXXTNEUVKFY,34,108,38635.01,544.0,20,60.92,80.16,4.9,...,0.39006,0.10963,0.55097,-0.56104,1,NaN,approve,0,7.9,NaN
1,CUST_FGUGTW230C,ACC_70VD7A4FFWCW,48,2,19912.97,703.0,21,112.11,571.12,0.3,...,0.03265,-0.40256,0.36218,0.86583,1,NaN,approve,0,5.5,NaN
2,CUST_8ZI3LCBZ0W,ACC_AF53381QSC0L,27,0,20326.87,720.0,25,73.61,492.57,4.6,...,-0.15637,0.57818,0.28902,-2.19864,1,NaN,approve,0,7.2,NaN
3,CUST_5MP3AR41CJ,ACC_U7WZGJ486LIV,45,49,38452.47,703.0,17,47.53,204.18,25.3,...,-1.02145,0.63908,-0.89190,-0.81592,1,NaN,approve,0,4.4,NaN
4,CUST_GNPL83JB0J,ACC_XW7DS3ED5J4Y,37,46,NaN,594.0,13,99.95,734.09,12.8,...,-0.65771,0.08020,0.17606,0.86739,1,NaN,approve,0,4.9,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159995,CUST_I81IW5SVRQ,ACC_UPDTFTYTSM0A,56,0,34775.62,727.0,21,51.72,226.11,3.8,...,-2.54086,-0.60747,0.23252,-0.06215,1,-1.535486,approve,0,1.0,NaN
159996,CUST_QT6DDEMKTJ,ACC_97NE0LBL5W9U,41,4,88617.57,770.0,18,NaN,171.07,15.1,...,0.34098,-1.78817,0.31788,0.51072,1,NaN,approve,0,7.4,NaN
159997,CUST_I0JS1GTS98,ACC_9JJ84W64Z7GX,30,2,41148.54,738.0,20,29.34,119.81,0.7,...,-1.28947,-0.32324,-0.06238,-0.99076,1,NaN,approve,3,6.6,NaN
159998,CUST_L7GUCJ3TFY,ACC_NGFXDR7HW1ZS,56,6,NaN,719.0,25,88.56,553.16,22.6,...,0.47179,-0.22090,-1.34239,-0.30513,1,NaN,approve,0,12.5,NaN


In [74]:
df_test = pd.read_csv("../data/kaggle_b2_fraud_test_v3.csv")

# Preprocessing Random Forest

In [75]:
customer_ids = df_test["customer_id"]


In [76]:
cols_to_drop = [
    "customer_id",
    "account_id",
    "customer_note",
    "last_ticket_subject",
    "manual_review_result",
    "post_event_status_code",
    "chargeback_resolution_time_days"
]

df = df.drop(columns=cols_to_drop, errors="ignore")


In [77]:
df["signup_date"] = pd.to_datetime(df["signup_date"])
df["account_age_days"] = (pd.Timestamp("today") - df["signup_date"]).dt.days
df = df.drop(columns=["signup_date"])

In [78]:
X = df.drop(columns=["target_is_fraud"])
y = df["target_is_fraud"]


In [79]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [80]:
num_cols = X_train.select_dtypes(include=["int64","float64"]).columns
cat_cols = X_train.select_dtypes(include=["object","category"]).columns


In [81]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [82]:
model = RandomForestClassifier(
    n_estimators=500,
    class_weight={0: 1, 1: 50},
    random_state=42,
    n_jobs=-1
)


In [83]:
clf = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model)
])


In [84]:
clf.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  Index(['age', 'tenure_months', 'annual_income_eur', 'credit_score',
       'num_transactions_30d', 'avg_amount_30d_eur', 'max_amount_30d_eur',
       'days_since_last_login', 'support_tickets_90d', 'chargebacks_12m',
       'failed_payments_6m', '...
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['channel', 'signup_source', 'plan_type', 'payment_method', 'browser',
       'os', 'occupation', 'device_type', 'merchant_category', 'country',
       'region', 'city', 'referrer_code', 'secondary_email'],
      dtype='object'))])),
                ('model',
                 RandomForestClassifier(class_weight={0: 1, 1: 50},
                                        n_estimators=500, n_jobs=-1,
                                        random_state=42))])

In [85]:
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98     31015
           1       1.00      0.02      0.03       985

    accuracy                           0.97     32000
   macro avg       0.98      0.51      0.51     32000
weighted avg       0.97      0.97      0.96     32000

ROC-AUC: 0.6300489283472628


In [86]:
cols_to_drop = [
    "customer_id",
    "account_id",
    "customer_note",
    "last_ticket_subject",
    "manual_review_result",
    "post_event_status_code",
    "chargeback_resolution_time_days"
]

df_test = df_test.drop(columns=cols_to_drop, errors="ignore")


In [87]:
df_test["signup_date"] = pd.to_datetime(df_test["signup_date"], errors="coerce")
df_test["account_age_days"] = (pd.Timestamp("today") - df_test["signup_date"]).dt.days
df_test = df_test.drop(columns=["signup_date"])


In [93]:
test_proba = clf.predict_proba(df_test)[:, 1]
test_pred = (test_proba > 0.05).astype(int)


In [94]:
submission = pd.DataFrame({
    "customer_id": customer_ids,
    "target_is_fraud": test_pred
})
submission


,customer_id,target_is_fraud
0,CUST_E5RX1BC9II,0
1,CUST_BHWIUKERGN,0
2,CUST_EXT9NA4CHU,0
3,CUST_9FSJE5R1NY,0
4,CUST_GDQXMODBED,0
...,...,...
39995,CUST_1OM9UCID91,0
39996,CUST_VDEY72BIZP,0
39997,CUST_UQEZ9KKIFG,0
39998,CUST_IXX0BE9VQD,0


In [95]:
submission.to_csv("../Preds/RandomForest_preds_Noah.csv", index=False)


In [96]:
print(y_train.mean())   # proportion de fraude
print(np.bincount(y_pred))


0.0307734375
[31984    16]


In [97]:
submission["target_is_fraud"].value_counts()

target_is_fraud
0    38734
1     1266
Name: count, dtype: int64

In [100]:
data.columns

Index(['customer_id', 'account_id', 'age', 'tenure_months',
       'annual_income_eur', 'credit_score', 'num_transactions_30d',
       'avg_amount_30d_eur', 'max_amount_30d_eur', 'days_since_last_login',
       'support_tickets_90d', 'chargebacks_12m', 'failed_payments_6m',
       'device_trust_z', 'ip_risk_z', 'is_vpn', 'num_devices_30d',
       'is_new_device', 'channel', 'signup_source', 'plan_type',
       'payment_method', 'browser', 'os', 'occupation', 'device_type',
       'merchant_category', 'country', 'region', 'city', 'postal_code',
       'last_ticket_subject', 'customer_note', 'referrer_code', 'signup_date',
       'secondary_email', 'target_is_fraud', 'income_log',
       'income_estimate_alt_eur', 'credit_score_norm',
       'tx_amount_total_30d_eur', 'max_to_avg_ratio', 'internal_signal_1',
       'internal_signal_2', 'internal_signal_3', 'internal_signal_4',
       'internal_signal_5', 'internal_signal_6', 'internal_signal_7',
       'internal_signal_8', 'terms_accepte